In [1]:
import os
import sys
import pandas as pd

sys.path.insert(0, os.path.join(os.path.abspath('.'), 'Target-Event-Agent_Networks'))
from teanets.nlp_utils import get_spacy_nlp
from teanets.svo_extraction import extract_svos

nlp = get_spacy_nlp()

def norm(x):
    s = str(x).strip().lower()
    return "__none__" if s in {"", "nan", "none"} else s

def prf(pred, gold):
    p = pred.apply(norm)
    g = gold.apply(norm)
    tp = (p == g).sum()
    n_pred = (p != "__none__").sum()
    n_gold = (g != "__none__").sum()
    precision = tp / n_pred if n_pred else 0.0
    recall = tp / n_gold if n_gold else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"TP": tp, "Pred": n_pred, "Gold": n_gold, "Precision": round(precision, 3), "Recall": round(recall, 3), "F1": round(f1, 3)}

[nltk_data] Downloading package wordnet to /home/seb/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
CORPUS_FILE = "data/sample_sentences.csv"

df_corpus = pd.read_csv(CORPUS_FILE)
if "sentence" not in df_corpus.columns:
    raise ValueError("Il corpus deve avere la colonna 'sentence'.")

wdw_rows = []
for i, s in enumerate(df_corpus["sentence"].fillna("")):
    df_rel = extract_svos(nlp(s))

    who_did = df_rel[(df_rel["TEA"] == "Who") & (df_rel["WDW2"] == "Did")] if not df_rel.empty else pd.DataFrame()
    did_what = df_rel[(df_rel["TEA"] == "Did") & (df_rel["WDW2"] == "What")] if not df_rel.empty else pd.DataFrame()

    who = did = what = None
    if not who_did.empty:
        who = who_did.iloc[0]["Node 1"]
        did = who_did.iloc[0]["Node 2"]
    if not did_what.empty:
        if did is None:
            did = did_what.iloc[0]["Node 1"]
        same_verb = did_what[did_what["Node 1"] == did]
        what = same_verb.iloc[0]["Node 2"] if not same_verb.empty else did_what.iloc[0]["Node 2"]

    wdw_rows.append({
        "sent_id": i,
        "sentence": s,
        "who": who,
        "did": did,
        "what": what,
    })

df_tea = pd.DataFrame(wdw_rows)
print(f"Frasi corpus: {len(df_corpus)}")
print(f"Triple estratte (righe): {len(df_tea)}")
df_tea[["sentence", "who", "did", "what"]].head(50)

Frasi corpus: 100
Triple estratte (righe): 100


,sentence,who,did,what
0,The cat chased the mouse.,cat,chase,mouse
1,John reads books.,john,read,book
2,Mary loves chocolate.,mary,love,chocolate
3,The dog barks loudly.,dog,bark loudly,NaN
4,The sun sets in the west.,sun,set,in west
5,Birds fly.,bird,fly,NaN
6,The student is reading a book.,student,read,book
7,They have finished their homework.,they,finish,their homework
8,He will eat the apple.,he,will eat,apple
9,The team had lost the game.,team,lose,game


In [3]:
GOLD_FILE = "data/gold_standard_svo.csv"
df_gold = pd.read_csv(GOLD_FILE)

def extract_svo_dep(doc):
    subj = verb = obj = None
    for token in doc:
        if token.dep_ == 'ROOT':
            verb = token.lemma_
            for child in token.children:
                if child.dep_ in ('nsubj', 'nsubjpass', 'csubj', 'csubjpass'):
                    subj = child.lemma_
                
                # Object candidates
                if child.dep_ in ('dobj', 'pobj', 'attr', 'acomp', 'ccomp'):
                    obj = child.lemma_
                elif child.dep_ == 'prep':
                    for p_child in child.children:
                        if p_child.dep_ == 'pobj':
                            obj = f"{child.lemma_} {p_child.lemma_}"
                elif child.dep_ == 'agent': # Passive agent "by ..."
                    for p_child in child.children:
                        if p_child.dep_ == 'pobj':
                            obj = f"by {p_child.lemma_}"
                elif child.dep_ == 'xcomp': # Infinitives "wants to learn"
                    for xc_child in child.children:
                        if xc_child.dep_ in ('dobj', 'pobj', 'attr'):
                            obj = xc_child.lemma_
            break
    return subj, verb, obj

gold_rows = []
for i, row in df_gold.iterrows():
    s = str(row["sentence"])
    if s == 'nan': continue
    doc = nlp(s)
    subj, verb, obj = extract_svo_dep(doc)
    gold_rows.append({
        "sent_id": row.get("sent_id", i),
        "sentence": row["sentence"],
        "pred_subj": subj,
        "pred_verb": verb,
        "pred_obj": obj,
        "gold_subject": row["subject"],
        "gold_verb": row["verb"],
        "gold_object": row["object"]
    })

df_eval = pd.DataFrame(gold_rows)
metrics = {
    "Subject": prf(df_eval["pred_subj"], df_eval["gold_subject"]),
    "Verb": prf(df_eval["pred_verb"], df_eval["gold_verb"]),
    "Object": prf(df_eval["pred_obj"], df_eval["gold_object"]),
}
df_metrics = pd.DataFrame(metrics).T
print("Validazione SVO Pura (Subject-Verb-Object) contro Golden Standard:\n")
print(f"Frasi valutate: {len(df_eval)}")
display(df_metrics)


Validazione SVO Pura (Subject-Verb-Object) contro Golden Standard:

Frasi valutate: 100


,TP,Pred,Gold,Precision,Recall,F1
Subject,100.0,100.0,100.0,1.000,1.000,1.000
Verb,100.0,100.0,100.0,1.000,1.000,1.000
Object,93.0,76.0,70.0,1.224,1.329,1.274


In [4]:
# ── Cell 4: Passive Voice Handling Validation ──────────────────────
# Validates whether the TEA library correctly re-maps passive constructions:
#   - Passive WITH agent: agent → Who, patient → What
#   - Passive WITHOUT agent: patient stays in Who

# Gold standard for passive voice TEA remapping (after library processing)
passive_gold = [
    # (sentence, expected_who, expected_did, expected_what)
    # --- Passive WITH agent (agent → Who, nsubjpass → What) ---
    ("The mouse was chased by the cat.", "cat", "chase", "mouse"),
    ("The book was written by John.", "john", "write", "book"),
    ("The car is fixed by the mechanic.", "mechanic", "fix", "car"),
    ("The letter was delivered by the postman.", "postman", "deliver", "letter"),
    ("The patient was helped by the doctor.", "doctor", "help", "patient"),
    ("The song was sung by the choir.", "choir", "sing", "song"),
    ("The house was built by the workers.", "worker", "build", "house"),
    ("The report was reviewed by the committee.", "committee", "review", "report"),
    ("The city was destroyed by the earthquake.", "earthquake", "destroy", "city"),
    ("The children were taught math by the tutor.", "tutor", "teach", "child"),
    ("The message was forwarded by the assistant.", "assistant", "forward", "message"),
    ("The bridge was damaged by the storm.", "storm", "damage", "bridge"),
    ("The game was won by the underdog.", "underdog", "win", "game"),
    ("She was given a promotion by her boss.", "boss", "give", "she"),
    ("The software was developed by a small team.", "team", "develop", "software"),
    # --- Passive WITHOUT agent (patient stays in Who) ---
    ("The window was broken.", "window", "break", None),
    ("The project is completed.", "project", "complete", None),
    ("The suspect has been arrested.", "suspect", "arrest", None),
    ("Mistakes were made.", "mistake", "make", None),
    ("The cake was eaten.", "cake", "eat", None),
    ("The door was left open.", "door", "leave", None),
    ("The files have been deleted.", "file", "delete", None),
    ("The proposal was rejected.", "proposal", "reject", None),
    ("The vaccine was distributed globally.", "vaccine", "distribute", None),
    ("The flowers were watered every morning.", "flower", "water", None),
    ("He was promoted to senior manager.", "he", "promote", None),
    ("Mountains can be seen from here.", "mountain", "see", None),
    ("The winner was announced at midnight.", "winner", "announce", None),
    ("English is spoken worldwide.", "english", "speak", None),
    ("The prisoners were released after the trial.", "prisoner", "release", None),
]

passive_rows = []
for sent, g_who, g_did, g_what in passive_gold:
    doc = nlp(sent)
    df_rel = extract_svos(doc)
    # Extract Who-Did and Did-What from TEA output
    who_did = df_rel[(df_rel['TEA'] == 'Who') & (df_rel['WDW2'] == 'Did')] if not df_rel.empty else pd.DataFrame()
    did_what = df_rel[(df_rel['TEA'] == 'Did') & (df_rel['WDW2'] == 'What')] if not df_rel.empty else pd.DataFrame()
    pred_who = who_did.iloc[0]['Node 1'] if not who_did.empty else None
    pred_did = who_did.iloc[0]['Node 2'] if not who_did.empty else (did_what.iloc[0]['Node 1'] if not did_what.empty else None)
    pred_what = did_what.iloc[0]['Node 2'] if not did_what.empty else None
    passive_rows.append({
        'sentence': sent,
        'pred_who': pred_who, 'gold_who': g_who,
        'pred_did': pred_did, 'gold_did': g_did,
        'pred_what': pred_what, 'gold_what': g_what,
    })

df_passive = pd.DataFrame(passive_rows)
passive_metrics = {
    'Who (Subject)': prf(df_passive['pred_who'], df_passive['gold_who']),
    'Did (Verb)': prf(df_passive['pred_did'], df_passive['gold_did']),
    'What (Object)': prf(df_passive['pred_what'], df_passive['gold_what']),
}
df_passive_metrics = pd.DataFrame(passive_metrics).T
print(f'Validazione Forma Passiva (TEA remapping) — {len(passive_gold)} frasi passive:\n')
display(df_passive_metrics)
print('\nDettaglio errori (se presenti):')
errors = df_passive[
    (df_passive['pred_who'].apply(norm) != df_passive['gold_who'].apply(norm)) |
    (df_passive['pred_did'].apply(norm) != df_passive['gold_did'].apply(norm)) |
    (df_passive['pred_what'].apply(norm) != df_passive['gold_what'].apply(norm))
]
if errors.empty:
    print('Nessun errore.')
else:
    display(errors)


Validazione Forma Passiva (TEA remapping) — 30 frasi passive:



,TP,Pred,Gold,Precision,Recall,F1
Who (Subject),28.0,30.0,30.0,0.933,0.933,0.933
Did (Verb),26.0,30.0,30.0,0.867,0.867,0.867
What (Object),26.0,19.0,15.0,1.368,1.733,1.529



Dettaglio errori (se presenti):


,sentence,pred_who,gold_who,pred_did,gold_did,pred_what,gold_what
13,She was given a promotion by her boss.,her boss,boss,give,give,she,she
14,The software was developed by a small team.,small team,team,develop,develop,software,software
23,The vaccine was distributed globally.,vaccine,vaccine,distribute globally,distribute,NaN,NaN
24,The flowers were watered every morning.,flower,flower,water every morning,water,every morning,NaN
25,He was promoted to senior manager.,he,he,promote,promote,to senior manager,NaN
26,Mountains can be seen from here.,mountain,mountain,can see,see,NaN,NaN
27,The winner was announced at midnight.,winner,winner,announce,announce,at midnight,NaN
28,English is spoken worldwide.,english,english,speak worldwide,speak,NaN,NaN
29,The prisoners were released after the trial.,prisoner,prisoner,release,release,after trial,NaN
